# Analysis of Emergency Obstetric Care (EmOC) in Accra
> Note: This notebook requires the [environment dependencies](requirements.txt) to be installed
> as well as either an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local instance of the ORS server.

## Model Summary:

This notebook provides the means to generate a dataset that is described in the [model documentation](../kano/dataset-interpretability.md).

## Workflow Summary:

The notebook gives an overview of the distribution of centres offering EmOC in the city, their classification and how they can be accessed during an emergency. Open source data from OpenStreetMap and tools (such as the openrouteservice) were used to create accessibility measures. Spatial analysis and other data analytics functions led to generating outputs within the 100x100m grid cells that categorised them into three levels: low, medium, and high.

* **Preprocessing**: Get data for EmOC facilities.
* **Analysis for Offer**:
    * Filter or classify EmOC facilities based on discussed criteria.
    * Visualise EmOC faccilities in their categories.
* **Analysis for Accessibility**:
    * Compute travel times to facilities using openrouteservice API or other routing services.
    * Generate areas for low, medium and high categories based on discussed criteria.
* **Analysis for Demmand**:
    * Downscale the popluation data to the 100x100m grid cells.
    * Derive socio-economic descriptors based on discussed criteria.

* **Result**: Generate results as GIS-compatible files.


### Datasets and Tools:
* [openrouteservice](https://openrouteservice.org/) - generate isochrones on the OpenStreetMap road network

#  Workflow

Make sure you have the required packages installed. You can install them using pip:

```bash
pip install -r requirements.txt
```

This study integrates various Python geospatial analysis libraries and packages to support spatial data processing, visualization, and isochrone generation. The os module is used to interact with the operating system, managing file paths and reading environment variables such as API keys. folium library along with its MarkerCluster plugin, facilitates the creation of interactive maps for visualizing large-scale geospatial data. The openrouteservice.client serves as an interface to the OpenRouteService API, enabling the extraction of isochrones. pandas library for data analysis, provides functions for analyzing, cleaning, exploring, and manipulating data, while fiona supports reading and writing real-world data using multi-layered GIS formats, such as shapefiles. The shapely package is employed for the manipulation and analysis of planar geometric objects.

## Setting up the virtual environment

```bash
# Create a new virtual environment
python -m venv .venv
activate .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [1]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd


import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point

from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

### Setting up the public API Key from OpenRouteService
In this study, users must obtain an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.


### Option 1: Using an ORS API Key
Make sure you have a .env file in the root directory with the following content:
```bash
    OPENROUTESERVICE_API_KEY='your_api_key'
```

In [ ]:
# Read the api key from the .env file
%load_ext dotenv
%dotenv
api_key = os.getenv('OPENROUTESERVICE_API_KEY')
client = openrouteservice.Client(key=api_key)

### Setting up relevant processing folders

There are different data sources used across the notebook. To handle these data sets, it is recommended to use three directories for input, temp and output data. Some of the files are related to healthcare facilities, population data. The healthcare facilities data is usualy the result of gathering global or national datasets and then carrying out local validation according to the local context. 

Despite being official, administrative boundaries may not reflect the actual patterns of human settlement or economic activity. Therefore, the team used the Functional Urban Area (FUA) as a complementary definition of the study areas. The FUA is defined by [the Joint Research Centre of the European Commission](https://commission.europa.eu/about/departments-and-executive-agencies/joint-research-centre_en) as the actual urban sprawl and human activities, encompassing the core city and economically or socially integrated surrounding regions. The FUA was obtained from [the Global Human Settlement Layer (GHSL) ](https://human-settlement.emergency.copernicus.eu/)dataset, which provides spatial data for functional urban areas worldwide. 

The following datasets are considered as input data for the analysis:


* [Datasets of health facilities](../scripts/Kano/data-inputs/healthcare_facilities.geojson)
* [Population: Women in childbearing age](../scripts/Kano/data-inputs/kano_nga_f_15_49_2015_1km.tif) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447)
* [Study Area](../../../docs/study-areas/grid-boundary-kano.gpkg) defined by the IDEAMAPS team

In [2]:
# Set paths to access Kano data
# Define directories
data_inputs = '../scripts/Accra/data-inputs/'
data_temp = '../scripts/Accra/data-temp/'
model_outputs = '../Accra/'

## 1. Data Collection

### Validated healthcare facilities - (Supply/Offer)
For Kano, the classification for validation was determined with the assistance of local experts, based on data obtained from the [datasets of health facilities](https://doi.org/10.6084/m9.figshare.22689667.v2).

In [5]:
healthcare_facilities_validated = gpd.read_file(data_inputs + 'healthcare_facilities_emoc.geojson')
healthcare_facilities_validated

,Region,District,FacilityName,Type,Town,Ownership,Latitude,Longitude,geometry
0,Greater Accra,Ga East,Achimota Hospital,Hospital,Atomic,Government,5.629220,-0.214760,POINT (-0.21476 5.62922)
1,Greater Accra,Accra Metropolitan,Adabraka Polyclinic,Polyclinic,Adabraka,Government,5.561290,-0.204770,POINT (-0.20477 5.56129)
2,Greater Accra,Tema Metropolitan,Afenyo Memorial Hospital,Hospital,Ashaiman,Private,5.690280,-0.034170,POINT (-0.03417 5.69028)
3,Greater Accra,Accra Metropolitan,Al-ayar Clinic & Maternity Home,Hospital,Akweteman,Private,5.613710,-0.240460,POINT (-0.24046 5.61371)
4,Greater Accra,Ga West,Amoah Memorial Hospital,Hospital,Awoshie (last stop),Private,5.590550,-0.287320,POINT (-0.28732 5.59055)
...,...,...,...,...,...,...,...,...,...
96,Greater Accra,Accra Metropolitan,Obenfo Hospital,Hospital,Mataheko,Private,5.580728,-0.210467,POINT (-0.21047 5.58073)
97,Greater Accra,Accra Metropolitan,Achimota Hospital,Hospital,Achimota,Government,5.580728,-0.210467,POINT (-0.21047 5.58073)
98,Greater Accra,Ga East,Van Medical Centre,Hospital,Ashongman,Private,5.708873,-0.243231,POINT (-0.24323 5.70887)
99,Greater Accra,Ga East,Jilac Specialist Hospital,Hospital,Dome,Private,5.708873,-0.243231,POINT (-0.24323 5.70887)


In [6]:
healthcare_facilities_validated

type_map = {
    'Polyclinic': 'Basic',
    'District Hospital': 'Comprehensive',
    'Hospital': 'Comprehensive'
}

owner_map = {
    'Government': 'Public',
    'Quasi-Government': 'Public',
    'Private': 'Private'
}

healthcare_facilities_validated['EmOC_Type'] = healthcare_facilities_validated['Type'].map(type_map)
healthcare_facilities_validated['Ownership_Type'] = healthcare_facilities_validated['Ownership'].map(owner_map).fillna('Private')

healthcare_facilities_validated['Local_Validation'] = np.where(
    healthcare_facilities_validated['EmOC_Type'].notna(),
    healthcare_facilities_validated['Ownership_Type'] + ' ' + healthcare_facilities_validated['EmOC_Type'] + ' EmOC',
    None
)

healthcare_facilities_validated


,Region,District,FacilityName,Type,Town,Ownership,Latitude,Longitude,geometry,EmOC_Type,Ownership_Type,Local_Validation
0,Greater Accra,Ga East,Achimota Hospital,Hospital,Atomic,Government,5.629220,-0.214760,POINT (-0.21476 5.62922),Comprehensive,Public,Public Comprehensive EmOC
1,Greater Accra,Accra Metropolitan,Adabraka Polyclinic,Polyclinic,Adabraka,Government,5.561290,-0.204770,POINT (-0.20477 5.56129),Basic,Public,Public Basic EmOC
2,Greater Accra,Tema Metropolitan,Afenyo Memorial Hospital,Hospital,Ashaiman,Private,5.690280,-0.034170,POINT (-0.03417 5.69028),Comprehensive,Private,Private Comprehensive EmOC
3,Greater Accra,Accra Metropolitan,Al-ayar Clinic & Maternity Home,Hospital,Akweteman,Private,5.613710,-0.240460,POINT (-0.24046 5.61371),Comprehensive,Private,Private Comprehensive EmOC
4,Greater Accra,Ga West,Amoah Memorial Hospital,Hospital,Awoshie (last stop),Private,5.590550,-0.287320,POINT (-0.28732 5.59055),Comprehensive,Private,Private Comprehensive EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...
96,Greater Accra,Accra Metropolitan,Obenfo Hospital,Hospital,Mataheko,Private,5.580728,-0.210467,POINT (-0.21047 5.58073),Comprehensive,Private,Private Comprehensive EmOC
97,Greater Accra,Accra Metropolitan,Achimota Hospital,Hospital,Achimota,Government,5.580728,-0.210467,POINT (-0.21047 5.58073),Comprehensive,Public,Public Comprehensive EmOC
98,Greater Accra,Ga East,Van Medical Centre,Hospital,Ashongman,Private,5.708873,-0.243231,POINT (-0.24323 5.70887),Comprehensive,Private,Private Comprehensive EmOC
99,Greater Accra,Ga East,Jilac Specialist Hospital,Hospital,Dome,Private,5.708873,-0.243231,POINT (-0.24323 5.70887),Comprehensive,Private,Private Comprehensive EmOC


In [7]:
healthcare_facilities_validated['hcf_id'] = range(len(healthcare_facilities_validated))
healthcare_facilities_validated

,Region,District,FacilityName,Type,Town,Ownership,Latitude,Longitude,geometry,EmOC_Type,Ownership_Type,Local_Validation,hcf_id
0,Greater Accra,Ga East,Achimota Hospital,Hospital,Atomic,Government,5.629220,-0.214760,POINT (-0.21476 5.62922),Comprehensive,Public,Public Comprehensive EmOC,0
1,Greater Accra,Accra Metropolitan,Adabraka Polyclinic,Polyclinic,Adabraka,Government,5.561290,-0.204770,POINT (-0.20477 5.56129),Basic,Public,Public Basic EmOC,1
2,Greater Accra,Tema Metropolitan,Afenyo Memorial Hospital,Hospital,Ashaiman,Private,5.690280,-0.034170,POINT (-0.03417 5.69028),Comprehensive,Private,Private Comprehensive EmOC,2
3,Greater Accra,Accra Metropolitan,Al-ayar Clinic & Maternity Home,Hospital,Akweteman,Private,5.613710,-0.240460,POINT (-0.24046 5.61371),Comprehensive,Private,Private Comprehensive EmOC,3
4,Greater Accra,Ga West,Amoah Memorial Hospital,Hospital,Awoshie (last stop),Private,5.590550,-0.287320,POINT (-0.28732 5.59055),Comprehensive,Private,Private Comprehensive EmOC,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,Greater Accra,Accra Metropolitan,Obenfo Hospital,Hospital,Mataheko,Private,5.580728,-0.210467,POINT (-0.21047 5.58073),Comprehensive,Private,Private Comprehensive EmOC,96
97,Greater Accra,Accra Metropolitan,Achimota Hospital,Hospital,Achimota,Government,5.580728,-0.210467,POINT (-0.21047 5.58073),Comprehensive,Public,Public Comprehensive EmOC,97
98,Greater Accra,Ga East,Van Medical Centre,Hospital,Ashongman,Private,5.708873,-0.243231,POINT (-0.24323 5.70887),Comprehensive,Private,Private Comprehensive EmOC,98
99,Greater Accra,Ga East,Jilac Specialist Hospital,Hospital,Dome,Private,5.708873,-0.243231,POINT (-0.24323 5.70887),Comprehensive,Private,Private Comprehensive EmOC,99


In [8]:
healthcare_facilities_validated.to_file(data_inputs + 'healthcare_facilities_accra_emoc.geojson', driver='GeoJSON')

In [18]:
healthcare_facilities_validated = gpd.read_file(data_inputs + 'healthcare_facilities_accra_emoc.geojson', driver='GeoJSON')

/opt/miniconda3/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(


### Population Grid Data (Demand)
This data originally comes as a grid (1km resolution) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447) to transform it into a 100x100m grid, we use a procedure explained below. 

Note: explain the process to scale down the population data. 
note: explain the rational for female population between 15-49 years old.

In [18]:
study_area = gpd.read_file(data_inputs + 'grid-boundary-accra.gpkg')
raster_path = data_inputs + 'gha_f_15_49_2015_1km.tif'

Clipping the population data to our study area

In [19]:
with rasterio.open(raster_path) as dataset:
    geometries = [study_area.geometry.unary_union.__geo_interface__]
    clipped_image, clipped_transform = mask(dataset, geometries, crop=True)
    band1 = clipped_image[0] # Read the first band of the raster

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_89635/2284915905.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometries = [study_area.geometry.unary_union.__geo_interface__]


In [20]:
out_meta = dataset.meta.copy()
out_meta.update({
        "height": clipped_image.shape[1],
        "width": clipped_image.shape[2],
        "transform": clipped_transform
    })

In [21]:
with rasterio.open(data_inputs + 'accra_gha_f_15_49_2015_1km.tif', "w", **out_meta) as dest:
    dest.write(clipped_image)

### Adding population data at 1km grid to 100m grid

In [13]:
# reading in geotiff file as numpy array
def read_tif(file: Path):
    if not file.exists():
        raise FileNotFoundError(f'File {file} not found')

    with rasterio.open(file) as dataset:
        arr = dataset.read()  # (bands X height X width)
        nodata = dataset.nodata
        transform = dataset.transform
        crs = dataset.crs

    # Replace NoData value with NaN
    if nodata is not None:
        arr[arr == nodata] = np.nan

    return arr.transpose((1, 2, 0)), transform, crs

def raster2vector(arr, transform, crs) -> gpd.GeoDataFrame:
    height, width, bands = arr.shape

    # Generate pixel coordinates
    geometries = []
    pixel_values = []

    for row in range(height):
        for col in range(width):
            x_min, y_max = transform * (col, row)  # Top-left corner
            x_max, y_min = transform * (col + 1, row + 1)  # Bottom-right corner

            pixel_value = arr[row, col].tolist()[0]  # Convert numpy array to list
            polygon = Polygon([(x_min, y_max), (x_max, y_max), (x_max, y_min), (x_min, y_min)])

            geometries.append(polygon)
            pixel_values.append(pixel_value)

    # Convert to DataFrame
    gdf = gpd.GeoDataFrame({'pop_grid_pop': pixel_values, 'geometry': geometries}, crs=crs)

    return gdf

epsg = 'EPSG:32632'

In [15]:
# Preparing grid
grid_file = data_inputs + 'grid-boundary-accra.gpkg'
grid = gpd.read_file(grid_file)
grid = grid.to_crs(epsg)
grid['grid_id'] = range(len(grid))
grid = grid[['grid_id', 'geometry','latitude', 'lat_min', 'lat_max', 'longitude', 'lon_min','lon_max']].set_geometry('geometry')
grid

,grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,"POLYGON ((-542846.596 658160.139, -542845.099 ...",5.875197,5.874792,5.875602,-0.379889,-0.380390,-0.379388
1,1,"POLYGON ((-542734.309 658158.24, -542732.812 6...",5.875197,5.874792,5.875602,-0.378888,-0.379389,-0.378387
2,2,"POLYGON ((-542622.022 658156.342, -542620.525 ...",5.875197,5.874792,5.875602,-0.377887,-0.378387,-0.377386
3,3,"POLYGON ((-542509.736 658154.444, -542508.239 ...",5.875197,5.874792,5.875602,-0.376886,-0.377386,-0.376385
4,4,"POLYGON ((-542397.45 658152.546, -542395.953 6...",5.875197,5.874792,5.875602,-0.375885,-0.376385,-0.375384
...,...,...,...,...,...,...,...,...
220278,220278,"POLYGON ((-547554.931 609298.045, -547553.543 ...",5.438454,5.438049,5.438860,-0.414731,-0.415232,-0.414231
220279,220279,"POLYGON ((-547442.596 609296.28, -547441.208 6...",5.438454,5.438049,5.438860,-0.413731,-0.414231,-0.413230
220280,220280,"POLYGON ((-547330.262 609294.515, -547328.874 ...",5.438454,5.438049,5.438860,-0.412730,-0.413231,-0.412230
220281,220281,"POLYGON ((-547217.927 609292.75, -547216.54 60...",5.438454,5.438049,5.438860,-0.411730,-0.412230,-0.411229


Building footprint data is used to estimate population distribution within each 1km cell. We recommend using open-source building footprint data from the [Overture Map Foundation](https://overturemaps.org/). Building centroids are spatially joined to a 100 m resolution grid, and the number of buildings within each 100 m cell (bcount) is subsequently calculated.

In [17]:
# Count buildings per grid cell

# Load Google building footprints
building_file = data_inputs + 'accra_GOB.parquet'
buildings = gpd.read_parquet(building_file)
buildings = buildings.to_crs(epsg)
buildings['centroid'] = buildings['geometry'].centroid

for df in [grid, buildings]:
    cols_to_remove = [c for c in df.columns if c.startswith('index_') or c.endswith('_left') or c.endswith('_right')]
    if cols_to_remove:
        df.drop(columns=cols_to_remove, inplace=True)

# Join buildings to grid using centroid
grid_buildings = grid.sjoin(
    buildings.set_geometry('centroid').drop(columns='geometry'),
    how='inner',
    predicate='intersects'
)

# Count buildings per grid cell
building_counts = grid_buildings.groupby('grid_id').size().rename('bcount')

# Add building count to grid
grid = grid.merge(building_counts, on='grid_id', how='left')
grid['bcount'] = grid['bcount'].fillna(0)   # assign 0 to empty cells
grid

,grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max,bcount
0,0,"POLYGON ((-542846.596 658160.139, -542845.099 ...",5.875197,5.874792,5.875602,-0.379889,-0.380390,-0.379388,0.0
1,1,"POLYGON ((-542734.309 658158.24, -542732.812 6...",5.875197,5.874792,5.875602,-0.378888,-0.379389,-0.378387,0.0
2,2,"POLYGON ((-542622.022 658156.342, -542620.525 ...",5.875197,5.874792,5.875602,-0.377887,-0.378387,-0.377386,0.0
3,3,"POLYGON ((-542509.736 658154.444, -542508.239 ...",5.875197,5.874792,5.875602,-0.376886,-0.377386,-0.376385,0.0
4,4,"POLYGON ((-542397.45 658152.546, -542395.953 6...",5.875197,5.874792,5.875602,-0.375885,-0.376385,-0.375384,0.0
...,...,...,...,...,...,...,...,...,...
220278,220278,"POLYGON ((-547554.931 609298.045, -547553.543 ...",5.438454,5.438049,5.438860,-0.414731,-0.415232,-0.414231,0.0
220279,220279,"POLYGON ((-547442.596 609296.28, -547441.208 6...",5.438454,5.438049,5.438860,-0.413731,-0.414231,-0.413230,0.0
220280,220280,"POLYGON ((-547330.262 609294.515, -547328.874 ...",5.438454,5.438049,5.438860,-0.412730,-0.413231,-0.412230,0.0
220281,220281,"POLYGON ((-547217.927 609292.75, -547216.54 60...",5.438454,5.438049,5.438860,-0.411730,-0.412230,-0.411229,0.0


The population of each 1km grid is distributed to underlying 100m cells proportionally based on building density. Each 100m grid is assigned a weight equal to its share of the total building count within the 1km grid.

In [22]:
# Adding population data at 1km grid to finer grid

data_path = Path(data_inputs)

# Load coarse population raster
pop_file = data_path / 'accra_gha_f_15_49_2015_1km.tif'
pop_raster, transform, crs = read_tif(pop_file)

# Convert raster to vector population grid
pop_grid = raster2vector(pop_raster, transform, crs)
pop_grid = pop_grid.to_crs(epsg)
pop_grid['pop_grid_id'] = range(len(pop_grid))
pop_grid.to_csv(data_path / 'pop_grid_id.csv')

# Assign coarse population data to finer grid based on the centroid locations of the finer grid cells
grid['centroid'] = grid['geometry'].centroid
grid = gpd.sjoin(grid.set_geometry('centroid'), pop_grid, how='left', predicate='within')
print(grid.columns)
grid = grid[['grid_id', 'bcount', 'pop_grid_id', 'geometry', 'latitude', 'lat_min', 'lat_max',
       'longitude', 'lon_min', 'lon_max']]
grid.head()

Index(['grid_id', 'geometry', 'latitude', 'lat_min', 'lat_max', 'longitude',
       'lon_min', 'lon_max', 'bcount', 'centroid', 'index_right',
       'pop_grid_pop', 'pop_grid_id'],
      dtype='object')


,grid_id,bcount,pop_grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,0.0,24,"POLYGON ((-542846.596 658160.139, -542845.099 ...",5.875197,5.874792,5.875602,-0.379889,-0.380390,-0.379388
1,1,0.0,24,"POLYGON ((-542734.309 658158.24, -542732.812 6...",5.875197,5.874792,5.875602,-0.378888,-0.379389,-0.378387
2,2,0.0,24,"POLYGON ((-542622.022 658156.342, -542620.525 ...",5.875197,5.874792,5.875602,-0.377887,-0.378387,-0.377386
3,3,0.0,24,"POLYGON ((-542509.736 658154.444, -542508.239 ...",5.875197,5.874792,5.875602,-0.376886,-0.377386,-0.376385
4,4,0.0,24,"POLYGON ((-542397.45 658152.546, -542395.953 6...",5.875197,5.874792,5.875602,-0.375885,-0.376385,-0.375384


In [23]:
# Calculate population weight (fraction of total population count that should be assigned to cell based on its building count)
grid_grouped_pop = grid.groupby('pop_grid_id')
building_count_pop = grid_grouped_pop['bcount'].sum().rename('pop_grid_bcount')
grid = grid.merge(building_count_pop, on='pop_grid_id', how='left')
grid['pop_weight'] = grid['bcount'] / grid['pop_grid_bcount']

# Compute disaggregated population count based on weight and building count at coarser cell level
grid = grid.merge(pop_grid, on='pop_grid_id', how='left')
grid['pop'] = grid['pop_grid_pop'] * grid['pop_weight']
grid.head()

,grid_id,bcount,pop_grid_id,geometry_x,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,geometry_y,pop
0,0,0.0,24,"POLYGON ((-542846.596 658160.139, -542845.099 ...",5.875197,5.874792,5.875602,-0.379889,-0.380390,-0.379388,41.0,0.0,NaN,"POLYGON ((-542951.301 658792.223, -542016.549 ...",NaN
1,1,0.0,24,"POLYGON ((-542734.309 658158.24, -542732.812 6...",5.875197,5.874792,5.875602,-0.378888,-0.379389,-0.378387,41.0,0.0,NaN,"POLYGON ((-542951.301 658792.223, -542016.549 ...",NaN
2,2,0.0,24,"POLYGON ((-542622.022 658156.342, -542620.525 ...",5.875197,5.874792,5.875602,-0.377887,-0.378387,-0.377386,41.0,0.0,NaN,"POLYGON ((-542951.301 658792.223, -542016.549 ...",NaN
3,3,0.0,24,"POLYGON ((-542509.736 658154.444, -542508.239 ...",5.875197,5.874792,5.875602,-0.376886,-0.377386,-0.376385,41.0,0.0,NaN,"POLYGON ((-542951.301 658792.223, -542016.549 ...",NaN
4,4,0.0,24,"POLYGON ((-542397.45 658152.546, -542395.953 6...",5.875197,5.874792,5.875602,-0.375885,-0.376385,-0.375384,41.0,0.0,NaN,"POLYGON ((-542951.301 658792.223, -542016.549 ...",NaN


In [24]:
# Saving to file
grid = grid.drop(columns=["geometry_y"])
grid.head()


,grid_id,bcount,pop_grid_id,geometry_x,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,pop
0,0,0.0,24,"POLYGON ((-542846.596 658160.139, -542845.099 ...",5.875197,5.874792,5.875602,-0.379889,-0.380390,-0.379388,41.0,0.0,NaN,NaN
1,1,0.0,24,"POLYGON ((-542734.309 658158.24, -542732.812 6...",5.875197,5.874792,5.875602,-0.378888,-0.379389,-0.378387,41.0,0.0,NaN,NaN
2,2,0.0,24,"POLYGON ((-542622.022 658156.342, -542620.525 ...",5.875197,5.874792,5.875602,-0.377887,-0.378387,-0.377386,41.0,0.0,NaN,NaN
3,3,0.0,24,"POLYGON ((-542509.736 658154.444, -542508.239 ...",5.875197,5.874792,5.875602,-0.376886,-0.377386,-0.376385,41.0,0.0,NaN,NaN
4,4,0.0,24,"POLYGON ((-542397.45 658152.546, -542395.953 6...",5.875197,5.874792,5.875602,-0.375885,-0.376385,-0.375384,41.0,0.0,NaN,NaN


In [ ]:
grid = grid.set_geometry("geometry_x")
grid = grid.to_crs(4326)
grid.to_file(data_temp + 'pop-grid-accra.gpkg', driver='GPKG')

In [28]:
# Preparing gird centroids with population attribute for accessibility analysis
grid = gpd.read_file(data_temp + "pop-grid-accra.gpkg")
grid_ll = grid.to_crs(epsg=4326)

grid_ll["geometry"] = grid_ll.geometry.centroid

grid_ll["latitude"] = grid_ll["geometry"].y
grid_ll["longitude"] = grid_ll["geometry"].x

grid_centroids = grid_ll[["grid_id", "latitude", "longitude", "geometry", "pop"]].copy()
grid_centroids = grid_centroids.set_geometry("geometry")
grid_centroids.set_crs("EPSG:4326", inplace=True)

grid_centroids = grid_centroids.dropna(subset=["pop"])
grid_centroids.reset_index(drop=True, inplace=True)

grid_centroids.to_file(data_temp + "grid_centroids.gpkg", driver="GPKG")

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_89635/3437270891.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid_ll["geometry"] = grid_ll.geometry.centroid


In [29]:
grid_centroids

,grid_id,latitude,longitude,geometry,pop
0,320,5.871955,-0.379887,POINT (-0.37989 5.87196),0.000000
1,321,5.871955,-0.378886,POINT (-0.37889 5.87196),0.000000
2,322,5.871955,-0.377885,POINT (-0.37789 5.87196),2.262094
3,323,5.871955,-0.376884,POINT (-0.37688 5.87196),0.205645
4,324,5.871955,-0.375883,POINT (-0.37588 5.87196),1.028224
...,...,...,...,...,...
205778,220188,5.439265,-0.434743,POINT (-0.43474 5.43926),0.000000
205779,220189,5.439265,-0.433742,POINT (-0.43374 5.43926),0.000000
205780,220190,5.439265,-0.432742,POINT (-0.43274 5.43926),0.000000
205781,220191,5.439265,-0.431741,POINT (-0.43174 5.43926),0.000000


## 2. Spatial Analysis Pipeline

### Travel time and dista calculation using OpenRouteService (ORS)

Using OpenRouteService (ORS) Matrix API to calculate the travel time and distance from each population grid centroid to the healthcare facility. There are two options to process the time and distance calculations: Using the public ORS API or using a local instance of the ORS server.

note: this will generate a file 'OD_matrix_healthcare_pop_grid‘

In [ ]:
origin_gdf = gpd.read_file(data_temp + "grid_centroids.gpkg")
origin_name_column = 'grid_id'
destination_gdf = gpd.read_file(data_inputs + 'healthcare_facilities_accra.geojson').dropna(subset=['geometry'])
destination_name_column = 'hcf_id'

In [4]:
# Extract coordinates
origins = list(zip(origin_gdf.geometry.x, origin_gdf.geometry.y))
destinations = list(zip(destination_gdf.geometry.x, destination_gdf.geometry.y))
locations = origins + destinations

In [5]:
# Indices
origins_index = list(range(0, len(origins)))
destinations_index = list(range(len(origins), len(locations)))

In [7]:
# Prepare API request
body = {
    'locations': locations,
    'destinations': destinations_index,
    'sources': origins_index,
    'metrics': ['distance', 'duration']
}

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': api_key,
    'Content-Type': 'application/json; charset=utf-8'
}

# Make request
response = requests.post(
    'https://api.openrouteservice.org/v2/matrix/driving-car',
    json=body,
    headers=headers
)

In [8]:
# Parse response
distances = response.json().get('distances', [])
durations = response.json().get('durations', [])

In [ ]:
distances_duration_matrix = []

# Iterate over each origin (grid)
for origin_index, origin in origin_gdf.iterrows():
    origin_name = origin[origin_name_column]
    origin_x = origin.geometry.x
    origin_y = origin.geometry.y
    origin_distances = distances[origin_index]
    origin_durations = durations[origin_index]

    # find the minimum duration and the index of the minimum duration
    min_duration = min(origin_durations)
    min_index = origin_durations.index(min_duration)
    destination_index = destinations_index[min_index]
    dest_x, dest_y = locations[destination_index]
    filtered = healthcare_facilities_validated[(destination_gdf.geometry.x == dest_x) & (destination_gdf.geometry.y == dest_y) ]
    destination_row = filtered.iloc[0]
    dest_name = destination_row[destination_name_column]

        # Append both the distance and duration for this origin-destination pair
    distances_duration_matrix.append([
            origin_name, origin_y, origin_x,
            dest_name, dest_y, dest_x,
            min_duration
        ])

In [ ]:
# Convert the results into a DataFrame
matrix_df = pd.DataFrame(distances_duration_matrix, columns=[
    'grid_code','origin_lat', 'origin_lon',
    'destination_name', 'dest_lat', 'dest_lon','min_duration'
])

In [ ]:
# Save to CSV
merged_df = pd.merge(matrix_df, grid_df[['grid_code', 'population']], on='grid_code', how='left')
merged_df.to_csv(data_temp + 'distance_duration_matrix_temp.csv', index=False)

In [ ]:
geometry = [Point(xy) for xy in zip(merged_df['dest_lon'], merged_df['dest_lat'])]
gdf = gpd.GeoDataFrame(merged_df, geometry=geometry, crs="EPSG:4326")

gpkg_path = data_temp + 'distance_duration_matrix_temp.gpkg'
gdf.to_file(gpkg_path, layer="duration_matrix", driver="GPKG")

### Option 2: Using a local ORS service
Make sure you have set a local service that runs the OSM-based ORS API. 
```r
# Insert R code from the local ORS service
```

### Procedure for Computing the OD Matrix Using a Local Docker Environment

This section outlines the steps required to compute the Origin-Destination (OD) matrix using a local Docker environment. 

1. **Set Up Docker Environment**:

2. **Prepare Input Data**:

3. **Run the OD Matrix Computation Script**:

4. **Monitor the Process**:

5. **Retrieve and Validate Output**:

### Diego please add description here

## Processing OD Matrix

Population data is the result of combining 1km grid data with 100m grid data. See [Section 2]() for more details.

In [3]:
# If not loaded yet, read from the temporary folder
centroids_df = gpd.read_file(data_temp +'pop-grid-accra.gpkg')
centroids_df

,grid_id,bcount,pop_grid_id,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,pop,geometry
0,0,0.0,24,5.875197,5.874792,5.875602,-0.379889,-0.380390,-0.379388,41.0,0.0,NaN,NaN,"POLYGON ((-0.37939 5.87479, -0.37939 5.8756, -..."
1,1,0.0,24,5.875197,5.874792,5.875602,-0.378888,-0.379389,-0.378387,41.0,0.0,NaN,NaN,"POLYGON ((-0.37839 5.87479, -0.37839 5.8756, -..."
2,2,0.0,24,5.875197,5.874792,5.875602,-0.377887,-0.378387,-0.377386,41.0,0.0,NaN,NaN,"POLYGON ((-0.37739 5.87479, -0.37739 5.8756, -..."
3,3,0.0,24,5.875197,5.874792,5.875602,-0.376886,-0.377386,-0.376385,41.0,0.0,NaN,NaN,"POLYGON ((-0.37639 5.87479, -0.37639 5.8756, -..."
4,4,0.0,24,5.875197,5.874792,5.875602,-0.375885,-0.376385,-0.375384,41.0,0.0,NaN,NaN,"POLYGON ((-0.37538 5.87479, -0.37538 5.8756, -..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220278,220278,0.0,4365,5.438454,5.438049,5.438860,-0.414731,-0.415232,-0.414231,0.0,NaN,NaN,NaN,"POLYGON ((-0.41423 5.43805, -0.41423 5.43886, ..."
220279,220279,0.0,4366,5.438454,5.438049,5.438860,-0.413731,-0.414231,-0.413230,0.0,NaN,NaN,NaN,"POLYGON ((-0.41323 5.43805, -0.41323 5.43886, ..."
220280,220280,0.0,4366,5.438454,5.438049,5.438860,-0.412730,-0.413231,-0.412230,0.0,NaN,NaN,NaN,"POLYGON ((-0.41223 5.43805, -0.41223 5.43886, ..."
220281,220281,0.0,4366,5.438454,5.438049,5.438860,-0.411730,-0.412230,-0.411229,0.0,NaN,NaN,NaN,"POLYGON ((-0.41123 5.43805, -0.41123 5.43886, ..."


In [4]:
# If not loaded yet, read from the temporary folder
matrix_df = pd.read_csv(data_temp +'OD-matrix-accra-access-emoc.csv')
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,1,320.0,2432.88,42.06
1,1,321.0,2432.88,42.06
2,1,322.0,2432.88,42.06
3,1,323.0,2442.29,42.10
4,1,324.0,2446.84,42.12
...,...,...,...,...
20784078,101,220188.0,3565.68,48.46
20784079,101,220189.0,3565.68,48.46
20784080,101,220190.0,3565.68,48.46
20784081,101,220191.0,3565.68,48.46


**GRID CELLS WITHOUT TRAVEL TIME ESTIMATE**

If a grid cell has a NULL value in the travel estimate, we will remove it from the analysis. This is because we cannot calculate the 2SFCA without a travel time estimate.

In [5]:
# Removing rows with NaN values in the 'duration_seconds' column
matrix_df = matrix_df.dropna(subset=['duration_seconds'])
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,1,320.0,2432.88,42.06
1,1,321.0,2432.88,42.06
2,1,322.0,2432.88,42.06
3,1,323.0,2442.29,42.10
4,1,324.0,2446.84,42.12
...,...,...,...,...
20784078,101,220188.0,3565.68,48.46
20784079,101,220189.0,3565.68,48.46
20784080,101,220190.0,3565.68,48.46
20784081,101,220191.0,3565.68,48.46


To process the OD Matrix we need merge it to create an integrated dataset that combines data from the healthcare facilities and population grid.For doing so, we will use the pandas library and join functions based on the id columns of all datasets.

In [10]:
pop_centroids_hcf = pd.merge(
    matrix_df,
    centroids_df[['grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min',
                  'lon_max', 'lat_max', 'bcount', 'pop_grid_bcount',
                  'pop_grid_pop', 'pop', 'geometry']],
    left_on='destination_id',
    right_on='grid_id',
    how='left'
)


In [11]:
pop_centroids_hcf

,origin_id,destination_id,duration_seconds,distance_km,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,bcount,pop_grid_bcount,pop_grid_pop,pop,geometry
0,1,320.0,2432.88,42.06,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,348.0,71.564423,0.000000,"POLYGON ((-0.37939 5.87155, -0.37939 5.87236, ..."
1,1,321.0,2432.88,42.06,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.0,348.0,71.564423,0.000000,"POLYGON ((-0.37839 5.87155, -0.37839 5.87236, ..."
2,1,322.0,2432.88,42.06,322,-0.377885,5.871955,-0.378386,5.87155,-0.377385,5.87236,11.0,348.0,71.564423,2.262094,"POLYGON ((-0.37738 5.87155, -0.37739 5.87236, ..."
3,1,323.0,2442.29,42.10,323,-0.376884,5.871955,-0.377385,5.87155,-0.376384,5.87236,1.0,348.0,71.564423,0.205645,"POLYGON ((-0.37638 5.87155, -0.37638 5.87236, ..."
4,1,324.0,2446.84,42.12,324,-0.375883,5.871955,-0.376384,5.87155,-0.375383,5.87236,5.0,348.0,71.564423,1.028224,"POLYGON ((-0.37538 5.87155, -0.37538 5.87236, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20784078,101,220188.0,3565.68,48.46,220188,-0.434743,5.439265,-0.435243,5.43886,-0.434243,5.43967,0.0,4.0,12.667079,0.000000,"POLYGON ((-0.43424 5.43886, -0.43424 5.43967, ..."
20784079,101,220189.0,3565.68,48.46,220189,-0.433742,5.439265,-0.434243,5.43886,-0.433242,5.43967,0.0,4.0,12.667079,0.000000,"POLYGON ((-0.43324 5.43886, -0.43324 5.43967, ..."
20784080,101,220190.0,3565.68,48.46,220190,-0.432742,5.439265,-0.433242,5.43886,-0.432241,5.43967,0.0,4.0,12.667079,0.000000,"POLYGON ((-0.43224 5.43886, -0.43224 5.43967, ..."
20784081,101,220191.0,3565.68,48.46,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.0,4.0,12.667079,0.000000,"POLYGON ((-0.43124 5.43886, -0.43124 5.43967, ..."


In [14]:
#pop_centroids_hcf = pd.merge(matrix_df, centroids_df[['rowid', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max', 'lat_max','bcount','pop_grid_bcount', 'pop_grid_pop', 'pop', 'geometry']], 
#                     left_on='destination_id', right_on='rowid', how='left')
#pop_centroids_hcf

In [15]:
pop_centroids_hcf.columns

Index(['origin_id', 'destination_id', 'duration_seconds', 'distance_km',
       'grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max',
       'lat_max', 'bcount', 'pop_grid_bcount', 'pop_grid_pop', 'pop',
       'geometry'],
      dtype='object')

In [16]:
pop_centroids_hcf = pop_centroids_hcf.rename(columns={
    "longitude": "origin_lon",
    "latitude": "origin_lat",
    "lon_min": "origin_lon_min",
    "lat_min": "origin_lat_min",
    "lon_max": "origin_lon_max",
    "lat_max": "origin_lat_max",
    #"rowid": "grid_id",
    "origin_id": "hcf_uid",
    "pop": "population"
})
columns_to_keep = ["grid_id", "origin_lon", "origin_lat", "origin_lon_min","origin_lat_min","origin_lon_max","origin_lat_max","population", "bcount","pop_grid_bcount", "pop_grid_pop","geometry", "hcf_uid", "duration_seconds", "distance_km"]
pop_centroids_hcf = pop_centroids_hcf[columns_to_keep]

In [17]:
pop_centroids_hcf

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,hcf_uid,duration_seconds,distance_km
0,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.000000,0.0,348.0,71.564423,"POLYGON ((-0.37939 5.87155, -0.37939 5.87236, ...",1,2432.88,42.06
1,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.000000,0.0,348.0,71.564423,"POLYGON ((-0.37839 5.87155, -0.37839 5.87236, ...",1,2432.88,42.06
2,322,-0.377885,5.871955,-0.378386,5.87155,-0.377385,5.87236,2.262094,11.0,348.0,71.564423,"POLYGON ((-0.37738 5.87155, -0.37739 5.87236, ...",1,2432.88,42.06
3,323,-0.376884,5.871955,-0.377385,5.87155,-0.376384,5.87236,0.205645,1.0,348.0,71.564423,"POLYGON ((-0.37638 5.87155, -0.37638 5.87236, ...",1,2442.29,42.10
4,324,-0.375883,5.871955,-0.376384,5.87155,-0.375383,5.87236,1.028224,5.0,348.0,71.564423,"POLYGON ((-0.37538 5.87155, -0.37538 5.87236, ...",1,2446.84,42.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20784078,220188,-0.434743,5.439265,-0.435243,5.43886,-0.434243,5.43967,0.000000,0.0,4.0,12.667079,"POLYGON ((-0.43424 5.43886, -0.43424 5.43967, ...",101,3565.68,48.46
20784079,220189,-0.433742,5.439265,-0.434243,5.43886,-0.433242,5.43967,0.000000,0.0,4.0,12.667079,"POLYGON ((-0.43324 5.43886, -0.43324 5.43967, ...",101,3565.68,48.46
20784080,220190,-0.432742,5.439265,-0.433242,5.43886,-0.432241,5.43967,0.000000,0.0,4.0,12.667079,"POLYGON ((-0.43224 5.43886, -0.43224 5.43967, ...",101,3565.68,48.46
20784081,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.000000,0.0,4.0,12.667079,"POLYGON ((-0.43124 5.43886, -0.43124 5.43967, ...",101,3565.68,48.46


Merging the dataframe than contains the od matrix (with the healthcare facility class) and the population data with the full information about health care facilities.

In [23]:
pop_centroids_hcf.columns

Index(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min',
       'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'population',
       'bcount', 'pop_grid_bcount', 'pop_grid_pop', 'geometry', 'hcf_uid',
       'duration_seconds', 'distance_km'],
      dtype='object')

In [22]:
healthcare_facilities_validated.columns

Index(['Region', 'District', 'FacilityName', 'Type', 'Town', 'Ownership',
       'Latitude', 'Longitude', 'EmOC_Type', 'Ownership_Type',
       'Local_Validation', 'hcf_id', 'geometry'],
      dtype='object')

In [24]:
distances_duration_matrix = pd.merge(pop_centroids_hcf, healthcare_facilities_validated[['hcf_id','FacilityName', 'Longitude', 'Latitude', 'Local_Validation']], 
                     left_on='hcf_uid', right_on='hcf_id', how='left')

In [25]:
distances_duration_matrix = distances_duration_matrix.rename(columns={
    "Longitude": "dest_lon",
    "Latitude": "dest_lat"
})
distances_duration_matrix = distances_duration_matrix.drop(columns=['hcf_uid'])

In [26]:
category_counts = healthcare_facilities_validated['Local_Validation'].value_counts()
print(category_counts)

Local_Validation
Private Comprehensive EmOC    77
Public Comprehensive EmOC     16
Public Basic EmOC              8
Name: count, dtype: int64


In [31]:
distances_duration_matrix['Local_Validation'].value_counts()

Local_Validation
Private Comprehensive EmOC    15845291
Public Comprehensive EmOC      3086745
Public Basic EmOC              1646264
Name: count, dtype: int64

In [32]:
distances_duration_matrix['Local_Validation'] = distances_duration_matrix['Local_Validation'].replace({
    'Public/Private Basic EmOC': 'Private Basic EmOC',
    'Public/Private comprehensive EmOC (missionary Hospital)': 'Private Comprehensive EmOC'
})

In [33]:
selected_categories = ['Public Comprehensive EmOC', 'Private Comprehensive EmOC', 
                       'Private Basic EmOC', 'Public Basic EmOC']

In [34]:
distances_duration_matrix = distances_duration_matrix[
    distances_duration_matrix['Local_Validation'].isin(selected_categories)]

distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,hcf_id,FacilityName,dest_lon,dest_lat,Local_Validation
0,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.000000,0.0,348.0,71.564423,"POLYGON ((-0.37939 5.87155, -0.37939 5.87236, ...",2432.88,42.06,1.0,Adabraka Polyclinic,-0.204770,5.561290,Public Basic EmOC
1,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.000000,0.0,348.0,71.564423,"POLYGON ((-0.37839 5.87155, -0.37839 5.87236, ...",2432.88,42.06,1.0,Adabraka Polyclinic,-0.204770,5.561290,Public Basic EmOC
2,322,-0.377885,5.871955,-0.378386,5.87155,-0.377385,5.87236,2.262094,11.0,348.0,71.564423,"POLYGON ((-0.37738 5.87155, -0.37739 5.87236, ...",2432.88,42.06,1.0,Adabraka Polyclinic,-0.204770,5.561290,Public Basic EmOC
3,323,-0.376884,5.871955,-0.377385,5.87155,-0.376384,5.87236,0.205645,1.0,348.0,71.564423,"POLYGON ((-0.37638 5.87155, -0.37638 5.87236, ...",2442.29,42.10,1.0,Adabraka Polyclinic,-0.204770,5.561290,Public Basic EmOC
4,324,-0.375883,5.871955,-0.376384,5.87155,-0.375383,5.87236,1.028224,5.0,348.0,71.564423,"POLYGON ((-0.37538 5.87155, -0.37538 5.87236, ...",2446.84,42.12,1.0,Adabraka Polyclinic,-0.204770,5.561290,Public Basic EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20578295,220188,-0.434743,5.439265,-0.435243,5.43886,-0.434243,5.43967,0.000000,0.0,4.0,12.667079,"POLYGON ((-0.43424 5.43886, -0.43424 5.43967, ...",3565.68,48.46,100.0,Holy Dove Hospital,-0.243231,5.708873,Private Comprehensive EmOC
20578296,220189,-0.433742,5.439265,-0.434243,5.43886,-0.433242,5.43967,0.000000,0.0,4.0,12.667079,"POLYGON ((-0.43324 5.43886, -0.43324 5.43967, ...",3565.68,48.46,100.0,Holy Dove Hospital,-0.243231,5.708873,Private Comprehensive EmOC
20578297,220190,-0.432742,5.439265,-0.433242,5.43886,-0.432241,5.43967,0.000000,0.0,4.0,12.667079,"POLYGON ((-0.43224 5.43886, -0.43224 5.43967, ...",3565.68,48.46,100.0,Holy Dove Hospital,-0.243231,5.708873,Private Comprehensive EmOC
20578298,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.000000,0.0,4.0,12.667079,"POLYGON ((-0.43124 5.43886, -0.43124 5.43967, ...",3565.68,48.46,100.0,Holy Dove Hospital,-0.243231,5.708873,Private Comprehensive EmOC


In [36]:
# creat subsets based on categories of 'Validation of HCFs Categorization'
categories = {
    "public_comprehensive_EmOC": ["Public Comprehensive EmOC"],
    "private_comprehensive_EmOC": ["Private Comprehensive EmOC"],
    "private_basic_EmOC": ["Private Basic EmOC"],
    "public_basic_EmOC": ["Public Basic EmOC"]
}

subsets = {
    key: distances_duration_matrix[
        distances_duration_matrix['Local_Validation'].str.contains('|'.join(values), na=False)
    ]
    for key, values in categories.items()
}

public_CEmOC = subsets["public_comprehensive_EmOC"]
private_CEmOC = subsets["private_comprehensive_EmOC"]
public_BEmOC = subsets["public_basic_EmOC"]
private_BEmOC = subsets["private_basic_EmOC"]

In [37]:
# Step 2: Define a function to get 3 smallest duration_seconds per grid_id for each category
def get_closest_3(df, n=3):
    return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)

In [38]:
# Step 3: If the subsets are already created for each category, we apply the function to each subset:
public_CEmOC_closest_3 = get_closest_3(public_CEmOC)
private_CEmOC_closest_3 = get_closest_3(private_CEmOC)
public_BEmOC_closest_3 = get_closest_3(public_BEmOC)
private_BEmOC_closest_3 = get_closest_3(private_BEmOC)

/var/folders/_0/jy_1jlp91v34q4g91twj01m80000gp/T/ipykernel_29630/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)
/var/folders/_0/jy_1jlp91v34q4g91twj01m80000gp/T/ipykernel_29630/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsm

In [39]:
# Step 4: Concatenate the filtered results into a single DataFrame
distances_duration_matrix = pd.concat([
    public_CEmOC_closest_3, private_CEmOC_closest_3,
    public_BEmOC_closest_3, private_BEmOC_closest_3
])
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,hcf_id,FacilityName,dest_lon,dest_lat,Local_Validation
0,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,71.564423,"POLYGON ((-0.37939 5.87155, -0.37939 5.87236, ...",2495.42,42.93,19.0,Dangme East District Hospital,0.569490,5.888880,Public Comprehensive EmOC
1,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,71.564423,"POLYGON ((-0.37939 5.87155, -0.37939 5.87236, ...",2560.25,43.38,31.0,Fire Medical Centre (Old site),-0.217700,5.532820,Public Comprehensive EmOC
2,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,71.564423,"POLYGON ((-0.37939 5.87155, -0.37939 5.87236, ...",2689.26,45.22,97.0,Achimota Hospital,-0.210467,5.580728,Public Comprehensive EmOC
3,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.0,0.0,348.0,71.564423,"POLYGON ((-0.37839 5.87155, -0.37839 5.87236, ...",2495.42,42.93,19.0,Dangme East District Hospital,0.569490,5.888880,Public Comprehensive EmOC
4,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.0,0.0,348.0,71.564423,"POLYGON ((-0.37839 5.87155, -0.37839 5.87236, ...",2560.25,43.38,31.0,Fire Medical Centre (Old site),-0.217700,5.532820,Public Comprehensive EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617344,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.0,0.0,4.0,12.667079,"POLYGON ((-0.43124 5.43886, -0.43124 5.43967, ...",2800.54,34.96,49.0,Maamobi Polyclinic,-0.199290,5.591840,Public Basic EmOC
617345,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.0,0.0,4.0,12.667079,"POLYGON ((-0.43124 5.43886, -0.43124 5.43967, ...",2960.96,39.55,1.0,Adabraka Polyclinic,-0.204770,5.561290,Public Basic EmOC
617346,220192,-0.430741,5.439265,-0.431241,5.43886,-0.430240,5.43967,0.0,0.0,4.0,12.667079,"POLYGON ((-0.43024 5.43886, -0.43024 5.43967, ...",2499.41,33.00,52.0,Mamprobi Polyclinic,-0.245560,5.538070,Public Basic EmOC
617347,220192,-0.430741,5.439265,-0.431241,5.43886,-0.430240,5.43967,0.0,0.0,4.0,12.667079,"POLYGON ((-0.43024 5.43886, -0.43024 5.43967, ...",2792.73,34.98,49.0,Maamobi Polyclinic,-0.199290,5.591840,Public Basic EmOC


In [40]:
geometry = [Point(xy) for xy in zip(distances_duration_matrix['origin_lon'], distances_duration_matrix['origin_lat'])]
gdf = gpd.GeoDataFrame(distances_duration_matrix, geometry=geometry, crs="EPSG:4326")

In [41]:
gpkg_path = data_temp + 'distances_duration_3_closet_Emoc.gpkg'
gdf.to_file(gpkg_path, layer="distances_duration_3_closet_Emoc", driver="GPKG")

In [42]:
# Review and remove
origin_dest = distances_duration_matrix

## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [43]:
# Function
from math import *
d = 10 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

78173.00674258533


In [44]:
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0      320   -0.379887    5.871955       -0.380388         5.87155   
1      320   -0.379887    5.871955       -0.380388         5.87155   
2      320   -0.379887    5.871955       -0.380388         5.87155   
3      321   -0.378886    5.871955       -0.379387         5.87155   
4      321   -0.378886    5.871955       -0.379387         5.87155   

   origin_lon_max  origin_lat_max  population  bcount  pop_grid_bcount  \
0       -0.379387         5.87236         0.0     0.0            348.0   
1       -0.379387         5.87236         0.0     0.0            348.0   
2       -0.379387         5.87236         0.0     0.0            348.0   
3       -0.378386         5.87236         0.0     0.0            348.0   
4       -0.378386         5.87236         0.0     0.0            348.0   

   pop_grid_pop                                           geometry  \
0     71.564423  POLYGON ((-0.37939 5.87155, -0.37939 5.87236, .

In [45]:
# Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [46]:
# Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')
origin_dest_acc = origin_dest  # Backup

In [47]:
# Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact.
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [48]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']

In [49]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,geometry,duration_seconds,distance_km,hcf_id,FacilityName,dest_lon,dest_lat,Local_Validation,Weight,Pop_W
0,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,...,"POLYGON ((-0.37939 5.87155, -0.37939 5.87236, ...",2495.42,42.93,19.0,Dangme East District Hospital,0.569490,5.888880,Public Comprehensive EmOC,0.0,0.0
1,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,...,"POLYGON ((-0.37939 5.87155, -0.37939 5.87236, ...",2560.25,43.38,31.0,Fire Medical Centre (Old site),-0.217700,5.532820,Public Comprehensive EmOC,0.0,0.0
2,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,...,"POLYGON ((-0.37939 5.87155, -0.37939 5.87236, ...",2689.26,45.22,97.0,Achimota Hospital,-0.210467,5.580728,Public Comprehensive EmOC,0.0,0.0
3,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.0,0.0,348.0,...,"POLYGON ((-0.37839 5.87155, -0.37839 5.87236, ...",2495.42,42.93,19.0,Dangme East District Hospital,0.569490,5.888880,Public Comprehensive EmOC,0.0,0.0
4,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.0,0.0,348.0,...,"POLYGON ((-0.37839 5.87155, -0.37839 5.87236, ...",2560.25,43.38,31.0,Fire Medical Centre (Old site),-0.217700,5.532820,Public Comprehensive EmOC,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617344,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.0,0.0,4.0,...,"POLYGON ((-0.43124 5.43886, -0.43124 5.43967, ...",2800.54,34.96,49.0,Maamobi Polyclinic,-0.199290,5.591840,Public Basic EmOC,0.0,0.0
617345,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.0,0.0,4.0,...,"POLYGON ((-0.43124 5.43886, -0.43124 5.43967, ...",2960.96,39.55,1.0,Adabraka Polyclinic,-0.204770,5.561290,Public Basic EmOC,0.0,0.0
617346,220192,-0.430741,5.439265,-0.431241,5.43886,-0.430240,5.43967,0.0,0.0,4.0,...,"POLYGON ((-0.43024 5.43886, -0.43024 5.43967, ...",2499.41,33.00,52.0,Mamprobi Polyclinic,-0.245560,5.538070,Public Basic EmOC,0.0,0.0
617347,220192,-0.430741,5.439265,-0.431241,5.43886,-0.430240,5.43967,0.0,0.0,4.0,...,"POLYGON ((-0.43024 5.43886, -0.43024 5.43967, ...",2792.73,34.98,49.0,Maamobi Polyclinic,-0.199290,5.591840,Public Basic EmOC,0.0,0.0


In [50]:
# Sum the Weighted Population
origin_dest_sum = origin_dest_acc.groupby(by='hcf_id')['Pop_W'].sum().reset_index()

In [51]:
origin_dest_sum

,hcf_id,Pop_W
0,1.0,10593.306432
1,2.0,17769.469723
2,3.0,16759.631699
3,4.0,13148.442517
4,5.0,10128.932122
...,...,...
93,95.0,17188.710302
94,96.0,5075.623285
95,97.0,34983.681351
96,99.0,3508.322208


In [52]:
# Merge the Sum of Weighted Population Back into the Original Data
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='hcf_id')

In [53]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,duration_seconds,distance_km,hcf_id,FacilityName,dest_lon,dest_lat,Local_Validation,Weight,Pop_W_x,Pop_W_y
0,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,...,2495.42,42.93,19.0,Dangme East District Hospital,0.569490,5.888880,Public Comprehensive EmOC,0.0,0.0,25969.200460
1,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,...,2560.25,43.38,31.0,Fire Medical Centre (Old site),-0.217700,5.532820,Public Comprehensive EmOC,0.0,0.0,18621.541874
2,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,...,2689.26,45.22,97.0,Achimota Hospital,-0.210467,5.580728,Public Comprehensive EmOC,0.0,0.0,34983.681351
3,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.0,0.0,348.0,...,2495.42,42.93,19.0,Dangme East District Hospital,0.569490,5.888880,Public Comprehensive EmOC,0.0,0.0,25969.200460
4,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.0,0.0,348.0,...,2560.25,43.38,31.0,Fire Medical Centre (Old site),-0.217700,5.532820,Public Comprehensive EmOC,0.0,0.0,18621.541874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1852042,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.0,0.0,4.0,...,2800.54,34.96,49.0,Maamobi Polyclinic,-0.199290,5.591840,Public Basic EmOC,0.0,0.0,44822.122516
1852043,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.0,0.0,4.0,...,2960.96,39.55,1.0,Adabraka Polyclinic,-0.204770,5.561290,Public Basic EmOC,0.0,0.0,10593.306432
1852044,220192,-0.430741,5.439265,-0.431241,5.43886,-0.430240,5.43967,0.0,0.0,4.0,...,2499.41,33.00,52.0,Mamprobi Polyclinic,-0.245560,5.538070,Public Basic EmOC,0.0,0.0,34874.100307
1852045,220192,-0.430741,5.439265,-0.431241,5.43886,-0.430240,5.43967,0.0,0.0,4.0,...,2792.73,34.98,49.0,Maamobi Polyclinic,-0.199290,5.591840,Public Basic EmOC,0.0,0.0,44822.122516


In [54]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [55]:
supply_map = {
    'Public Comprehensive EmOC': 1,
    'Private Comprehensive EmOC': 0.7,
    'Public Basic EmOC': 0.5,
    'Private Basic EmOC': 0.35
}

In [56]:
origin_dest_acc['supply'] = origin_dest_acc['Local_Validation'].map(supply_map)
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)

/var/folders/_0/jy_1jlp91v34q4g91twj01m80000gp/T/ipykernel_29630/2370967217.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)


In [57]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [58]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [59]:
# Normalize
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])

In [60]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,dest_lat,Local_Validation,Weight,Pop_W_x,Pop_W_S,supply,supply_demand_ratio,supply_W,Accessibility,Accessibility_standard
0,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,...,5.888880,Public Comprehensive EmOC,0.0,0.0,25969.200460,1.0,0.000039,0.0,0.0,0.0
1,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,...,5.532820,Public Comprehensive EmOC,0.0,0.0,18621.541874,1.0,0.000054,0.0,0.0,0.0
2,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,0.0,348.0,...,5.580728,Public Comprehensive EmOC,0.0,0.0,34983.681351,1.0,0.000029,0.0,0.0,0.0
3,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.0,0.0,348.0,...,5.888880,Public Comprehensive EmOC,0.0,0.0,25969.200460,1.0,0.000039,0.0,0.0,0.0
4,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.0,0.0,348.0,...,5.532820,Public Comprehensive EmOC,0.0,0.0,18621.541874,1.0,0.000054,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1852042,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.0,0.0,4.0,...,5.591840,Public Basic EmOC,0.0,0.0,44822.122516,0.5,0.000011,0.0,0.0,0.0
1852043,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.0,0.0,4.0,...,5.561290,Public Basic EmOC,0.0,0.0,10593.306432,0.5,0.000047,0.0,0.0,0.0
1852044,220192,-0.430741,5.439265,-0.431241,5.43886,-0.430240,5.43967,0.0,0.0,4.0,...,5.538070,Public Basic EmOC,0.0,0.0,34874.100307,0.5,0.000014,0.0,0.0,0.0
1852045,220192,-0.430741,5.439265,-0.431241,5.43886,-0.430240,5.43967,0.0,0.0,4.0,...,5.591840,Public Basic EmOC,0.0,0.0,44822.122516,0.5,0.000011,0.0,0.0,0.0


In [61]:
max(origin_dest_acc.Accessibility_standard)

1.0

In [62]:
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'acc_score_3closest.gpkg'
gdf.to_file(gpkg_path, layer="acc_score_3closest", driver="GPKG")

# 4. Grouping by grid ID to prepare the final output file
There is a need to update this part of the code

In [64]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'acc_score_3closest.gpkg')

In [65]:
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry']]

In [66]:
# Group by multiple columns and calculate the mean for numeric columns
# results_grid = results_grid.groupby(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard']).count().reset_index()
results_grid = results_grid.drop_duplicates(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry'])
type(results_grid)

geopandas.geodataframe.GeoDataFrame

In [67]:
# save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access', driver='GPKG')

In [68]:
results_grid

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Accessibility_standard,geometry
0,320,-0.379887,5.871955,-0.380388,5.87155,-0.379387,5.87236,0.0,"POLYGON ((-0.37939 5.87155, -0.37939 5.87236, ..."
3,321,-0.378886,5.871955,-0.379387,5.87155,-0.378386,5.87236,0.0,"POLYGON ((-0.37839 5.87155, -0.37839 5.87236, ..."
6,322,-0.377885,5.871955,-0.378386,5.87155,-0.377385,5.87236,0.0,"POLYGON ((-0.37738 5.87155, -0.37739 5.87236, ..."
9,323,-0.376884,5.871955,-0.377385,5.87155,-0.376384,5.87236,0.0,"POLYGON ((-0.37638 5.87155, -0.37638 5.87236, ..."
12,324,-0.375883,5.871955,-0.376384,5.87155,-0.375383,5.87236,0.0,"POLYGON ((-0.37538 5.87155, -0.37538 5.87236, ..."
...,...,...,...,...,...,...,...,...,...
617334,220188,-0.434743,5.439265,-0.435243,5.43886,-0.434243,5.43967,0.0,"POLYGON ((-0.43424 5.43886, -0.43424 5.43967, ..."
617337,220189,-0.433742,5.439265,-0.434243,5.43886,-0.433242,5.43967,0.0,"POLYGON ((-0.43324 5.43886, -0.43324 5.43967, ..."
617340,220190,-0.432742,5.439265,-0.433242,5.43886,-0.432241,5.43967,0.0,"POLYGON ((-0.43224 5.43886, -0.43224 5.43967, ..."
617343,220191,-0.431741,5.439265,-0.432242,5.43886,-0.431241,5.43967,0.0,"POLYGON ((-0.43124 5.43886, -0.43124 5.43967, ..."


### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  

In [69]:
results_grid['result'] = -1
results_grid.loc[results_grid['Accessibility_standard'] > 0.000001, 'result'] = 2
results_grid.loc[results_grid['Accessibility_standard'] > 0.005, 'result'] = 1
results_grid.loc[results_grid['Accessibility_standard'] > 0.02, 'result'] = 0

In [70]:
category_counts = results_grid['result'].value_counts()
print(category_counts)

result
-1    115556
 2     57987
 1     25280
 0      6960
Name: count, dtype: int64


### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [72]:
results_grid.columns

Index(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min',
       'origin_lat_min', 'origin_lon_max', 'origin_lat_max',
       'Accessibility_standard', 'geometry', 'result'],
      dtype='object')

In [73]:
results_grid['focused'] = 0
# Focus areas between the Low category and the excluded cells due to low population or no buildings
results_grid.loc[(results_grid['Accessibility_standard'] > 0.000001) & (results_grid['Accessibility_standard'] < 0.0000015), 'focused'] = 1
# Focus areas between the Medium and High categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.003) & (results_grid['Accessibility_standard'] < 0.006), 'focused'] = 1
# Focus areas between the Low and Medium categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.019) & (results_grid['Accessibility_standard'] < 0.03), 'focused'] = 1

In [75]:
category_counts = results_grid['focused'].value_counts()
print(category_counts)

focused
0    190070
1     15713
Name: count, dtype: int64


In [76]:
results_grid = results_grid.loc[results_grid['result'] != -1]

In [77]:
results_grid = results_grid.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max'
})

In [78]:
results_grid

,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,Accessibility_standard,geometry,result,focused
15084,5836,-0.173672,5.844402,-0.174172,5.843997,-0.173171,5.844807,0.000001,"POLYGON ((-0.17317 5.844, -0.17317 5.84481, -0...",2,1
15765,6065,-0.174673,5.843591,-0.175173,5.843186,-0.174172,5.843997,0.000001,"POLYGON ((-0.17417 5.84319, -0.17417 5.844, -0...",2,1
15768,6066,-0.173672,5.843591,-0.174172,5.843186,-0.173171,5.843997,0.000001,"POLYGON ((-0.17317 5.84319, -0.17317 5.844, -0...",2,1
17331,6595,-0.174672,5.841971,-0.175173,5.841565,-0.174172,5.842376,0.000001,"POLYGON ((-0.17417 5.84157, -0.17417 5.84238, ...",2,1
18150,6874,-0.175673,5.841160,-0.176174,5.840755,-0.175173,5.841565,0.000001,"POLYGON ((-0.17517 5.84076, -0.17517 5.84157, ...",2,1
...,...,...,...,...,...,...,...,...,...,...,...
565875,199013,-0.259665,5.513802,-0.260166,5.513397,-0.259165,5.514207,0.003903,"POLYGON ((-0.25916 5.5134, -0.25916 5.51421, -...",2,1
565878,199014,-0.258664,5.513802,-0.259165,5.513397,-0.258164,5.514207,0.003834,"POLYGON ((-0.25816 5.5134, -0.25816 5.51421, -...",2,1
565881,199015,-0.257664,5.513802,-0.258164,5.513397,-0.257163,5.514207,0.003796,"POLYGON ((-0.25716 5.5134, -0.25716 5.51421, -...",2,1
565884,199016,-0.256663,5.513802,-0.257164,5.513397,-0.256163,5.514207,0.003768,"POLYGON ((-0.25616 5.5134, -0.25616 5.51421, -...",2,1


In [79]:
# Save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access-class.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access-class', driver='GPKG')

In [84]:
# Save the results to a CSV file in the format required by the IDEAMAPS data ecosystem
results_table = results_grid.drop(columns=['Accessibility_standard', 'grid_id', 'geometry'])
results_table.to_csv(model_outputs + 'model-output.csv', index=False)